In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import pipeline


In [ ]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 5.8 MB/s eta 0:00:00


In [ ]:
# 1. Load PDF
loader = PyPDFLoader("Data.pdf")
documents = loader.load()

In [ ]:
# 2. Split text
text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# 4. Store in FAISS
db = FAISS.from_documents(docs, embedding)

In [ ]:
while True:
    query = input("\nAsk question (type 'exit'): ")

    if query.lower() == "exit":
        break

    # 5. Retrieve relevant chunks
    results = db.similarity_search(query, k=3)

    if not results:
        print("No relevant data found.")
        continue

    # 6. Combine context
    context = " ".join([doc.page_content for doc in results])
    context = context[:1000]  # limit size

    # 7. Create prompt
    prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}
"""

    # 8. Generate answer
    response = generate_answer(prompt)

    print("\nAnswer:\n", response)


Ask question (type 'exit'): Tomato__Tomato_mosaic_virus

Answer:
 The American Bollworm is a moth in its adult stage, but the larvae (caterpillars) are the primary concern for crops. These larvae are highly destructive, feeding on the cotton bolls, flowers, and leaves.

Ask question (type 'exit'): exit
